# econchile — charts tutorial

**Learn econchile v0.2.0 by making charts.** This notebook fetches Chilean macroeconomic data and visualizes it with matplotlib. You'll see 4 chart types (line, dual-axis, bar, multi-series) and learn how to handle the live API + offline fallback.

**How to run:**
```bash
pip install "econchile[examples]"
```

**Token optional:** Set `BCCH_TOKEN` for live data from the Banco Central de Chile API (free at https://si3.bcentral.cl/Siete/en/Siete/API). Without a token, the notebook uses the bundled offline fixture (`data/macro-chile-lean.csv`) — all charts still render.

In [ ]:
import os
import csv
import re
from collections import OrderedDict

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import econchile
from econchile import BcchClient, Series

print(f'econchile {econchile.__version__}')
print(f'matplotlib {matplotlib.__version__}')
print(f'Token: {"yes" if os.environ.get("BCCH_TOKEN") else "no (offline fallback)"}')

In [ ]:
client = BcchClient()

if os.environ.get('BCCH_TOKEN'):
    result = client.get(Series.USD, '2000-01-01', '2010-12-31')
    print(f'USD/CLP 2000-2010: {len(result.observations)} daily observations')
    print(f'Source: {result.source}')
    print('First 3 observations:')
    for obs in result.observations[:3]:
        print(f'  {obs.date}  {obs.value}')
else:
    print('No token — will use offline fixture for charts.')
    print('Set BCCH_TOKEN and re-run for live data.')

## Chart 1: USD/CLP line chart (2000–2010)

Daily exchange rate aggregated to month-end values (last observation of each month). Live API or offline CSV fallback.

In [ ]:
HERE = os.path.dirname(os.path.abspath('__file__'))
REPO_ROOT = os.path.dirname(HERE) if 'examples' in HERE else '.'
CSV_PATH = os.path.join(REPO_ROOT, 'data', 'macro-chile-lean.csv')

def _parse_period(text):
    m = re.match(r'([a-z]+)\.(\d{4})', text.strip().lower())
    if not m:
        return None
    months = {
        'ene': 1, 'feb': 2, 'mar': 3, 'abr': 4, 'may': 5, 'jun': 6,
        'jul': 7, 'ago': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dic': 12,
    }
    month = months.get(m.group(1))
    if month is None:
        return None
    return (int(m.group(2)), month)

def _parse_number(text):
    if text is None:
        return None
    cleaned = text.strip().replace('\xa0', '').replace(' ', '')
    if not cleaned:
        return None
    cleaned = cleaned.replace('.', '').replace(',', '.')
    try:
        return float(cleaned)
    except ValueError:
        return None

def _load_offline_series(column='TCN'):
    out = []
    with open(CSV_PATH, encoding='utf-8-sig') as fh:
        for row in csv.DictReader(fh, delimiter=';'):
            period = _parse_period(row.get('Periodo', ''))
            value = _parse_number(row.get(column))
            if period and value is not None:
                out.append((period[0], period[1], value))
    return sorted(out)

def _monthly_from_daily(daily_obs):
    months = OrderedDict()
    for obs in daily_obs:
        if obs.value is None:
            continue
        y, m, _ = obs.date.split('-')
        key = (int(y), int(m))
        months[key] = float(obs.value)
    return [(y, m, v) for (y, m), v in months.items()]

if os.environ.get('BCCH_TOKEN'):
    daily = client.get(Series.USD, '2000-01-01', '2010-12-31').observations
    series = _monthly_from_daily(daily)
    source_label = 'BCCh API (live)'
else:
    series = _load_offline_series('TCN')
    source_label = 'Offline fixture (quarterly)'

window = [(y, m, v) for (y, m, v) in series if 2000 <= y <= 2010]
x = [f'{y}-{m:02d}' for y, m, _ in window]
y = [v for _, _, v in window]

fig, ax = plt.subplots(figsize=(10, 5.5), dpi=150)
ax.plot(x, y, color='#0b5394', linewidth=2)
ax.fill_between(range(len(x)), y, color='#0b5394', alpha=0.08)
ax.set_title(f'USD/CLP — monthly, 2000–2010 ({source_label})')
ax.set_ylabel('CLP per USD')
tick_step = max(1, len(x) // 10)
ax.set_xticks(range(0, len(x), tick_step))
ax.set_xticklabels([x[i] for i in range(0, len(x), tick_step)], rotation=45, fontsize=8)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()
print(f'{len(window)} observations, source: {source_label}')

## Chart 2: TPM vs IPC dual-axis

Left axis: TPM (monetary policy rate, daily level). Right axis: IPC_SAE (month-over-month %, monthly). Two series, two frequencies, two axes.

In [ ]:
if os.environ.get('BCCH_TOKEN'):
    tpm = client.get(Series.TPM, '2010-01-01', '2020-12-31').observations
    ipc = client.get(Series.IPC_SAE, '2010-01-01', '2020-12-31').observations
    
    tpm_monthly = _monthly_from_daily(tpm)
    ipc_vals = [(int(o.date.split('-')[0]), int(o.date.split('-')[1]), o.value)
                for o in ipc if o.value is not None]
    
    fig, ax1 = plt.subplots(figsize=(10, 5), dpi=150)
    
    x1 = [f'{y}-{m:02d}' for y, m, _ in tpm_monthly]
    y1 = [v for _, _, v in tpm_monthly]
    ax1.plot(x1, y1, color='#cc0000', linewidth=2, label='TPM (left)')
    ax1.set_ylabel('TPM (%)', color='#cc0000')
    ax1.tick_params(axis='y', labelcolor='#cc0000')
    
    ax2 = ax1.twinx()
    x2 = [f'{y}-{m:02d}' for y, m, _ in ipc_vals]
    y2 = [v for _, _, v in ipc_vals]
    ax2.plot(x2, y2, color='#0000cc', linewidth=2, label='IPC_SAE MoM% (right)')
    ax2.set_ylabel('IPC_SAE MoM %', color='#0000cc')
    ax2.tick_params(axis='y', labelcolor='#0000cc')
    
    fig.suptitle('TPM vs IPC_SAE — 2010–2020 (BCCh API)')
    tick_step = max(1, len(x1) // 10)
    ax1.set_xticks(range(0, len(x1), tick_step))
    ax1.set_xticklabels([x1[i] for i in range(0, len(x1), tick_step)], rotation=45, fontsize=8)
    ax1.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    print(f'TPM: {len(tpm_monthly)} monthly, IPC_SAE: {len(ipc_vals)} monthly')
else:
    offline = _load_offline_series('TPM')
    window = [(y, m, v) for (y, m, v) in offline if 2010 <= y <= 2020]
    x = [f'{y}-{m:02d}' for y, m, _ in window]
    y = [v for _, _, v in window]
    
    fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
    ax.plot(x, y, color='#cc0000', linewidth=2)
    ax.set_title('TPM — 2010–2020 (offline fixture, quarterly)')
    ax.set_ylabel('TPM (%)')
    tick_step = max(1, len(x) // 10)
    ax.set_xticks(range(0, len(x), tick_step))
    ax.set_xticklabels([x[i] for i in range(0, len(x), tick_step)], rotation=45, fontsize=8)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    print(f'TPM offline: {len(window)} quarterly observations')

## Chart 3: PIB bar chart (quarterly)

GDP (PIB) chained volumes, quarterly. Bar chart with rotated x-labels.

In [ ]:
if os.environ.get('BCCH_TOKEN'):
    pib = client.get(Series.PIB, '2010-01-01', '2023-12-31').observations
    pib_vals = [(int(o.date.split('-')[0]), int(o.date.split('-')[1]), o.value)
                for o in pib if o.value is not None]
else:
    pib_vals = _load_offline_series('PIB')

window = [(y, m, v) for (y, m, v) in pib_vals if 2010 <= y <= 2023]
x = [f'{y}-Q{(m-1)//3+1}' for y, m, _ in window]
y = [v for _, _, v in window]

fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
ax.bar(range(len(x)), y, color='#3d85c6', width=0.8)
ax.set_title('PIB (quarterly, chained volumes) — 2010–2023')
ax.set_ylabel('PIB')
tick_step = max(1, len(x) // 12)
ax.set_xticks(range(0, len(x), tick_step))
ax.set_xticklabels([x[i] for i in range(0, len(x), tick_step)], rotation=45, fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()
print(f'{len(window)} quarterly observations')

## Chart 4: IMACEC original vs seasonally adjusted

IMACEC (monthly economic activity) has two versions: original (raw) and SA (seasonally adjusted). The original shows December spikes; the SA version smooths seasonal patterns. **Live API only** — offline fixture doesn't include IMACEC.

In [ ]:
if os.environ.get('BCCH_TOKEN'):
    imacec = client.get(Series.IMACEC, '2015-01-01', '2023-12-31').observations
    imacec_sa = client.get(Series.IMACEC_SA, '2015-01-01', '2023-12-31').observations
    
    orig = [(int(o.date.split('-')[0]), int(o.date.split('-')[1]), o.value)
            for o in imacec if o.value is not None]
    sa = [(int(o.date.split('-')[0]), int(o.date.split('-')[1]), o.value)
          for o in imacec_sa if o.value is not None]
    
    fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
    
    x1 = [f'{y}-{m:02d}' for y, m, _ in orig]
    y1 = [v for _, _, v in orig]
    ax.plot(x1, y1, color='#999999', linewidth=1.5, alpha=0.7, label='IMACEC (original)')
    
    x2 = [f'{y}-{m:02d}' for y, m, _ in sa]
    y2 = [v for _, _, v in sa]
    ax.plot(x2, y2, color='#0b5394', linewidth=2, label='IMACEC_SA (adjusted)')
    
    ax.set_title('IMACEC: original vs seasonally adjusted — 2015–2023')
    ax.set_ylabel('IMACEC index')
    ax.legend(loc='upper left')
    tick_step = max(1, len(x1) // 10)
    ax.set_xticks(range(0, len(x1), tick_step))
    ax.set_xticklabels([x1[i] for i in range(0, len(x1), tick_step)], rotation=45, fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    plt.show()
    print(f'IMACEC original: {len(orig)}, SA: {len(sa)} monthly observations')
else:
    print('IMACEC not available in offline fixture.')
    print('Set BCCH_TOKEN to see the original vs seasonally adjusted comparison.')

## Offline fallback: bundled CSV fixture

When `BCCH_TOKEN` is missing or the API is down, the notebook falls back to `data/macro-chile-lean.csv` (quarterly data, 1996–2023). Same parsing helpers as `charts_demo.py`.

In [ ]:
offline_data = _load_offline_series('TCN')
print(f'Offline fixture: {len(offline_data)} quarterly observations')
print(f'First: {offline_data[0]}')
print(f'Last: {offline_data[-1]}')

x = [f'{y}-{m:02d}' for y, m, _ in offline_data]
y = [v for _, _, v in offline_data]

fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
ax.plot(x, y, color='#0b5394', linewidth=2)
ax.set_title('TCN (nominal exchange rate) — offline fixture, 1996–2023')
ax.set_ylabel('CLP per USD')
tick_step = max(1, len(x) // 10)
ax.set_xticks(range(0, len(x), tick_step))
ax.set_xticklabels([x[i] for i in range(0, len(x), tick_step)], rotation=45, fontsize=8)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

## Pitfalls & tips

- **ND → None:** BCCh marks missing data with `statusCode == "ND"` → parsed as `value=None`, never zero.
- **EURO ≠ CLP/EUR:** `Series.EURO` is USD per EUR, not CLP per EUR.
- **Frequencies:** Daily (UF, USD, TPM), monthly (IPC, IMACEC), quarterly (PIB), annual (PIB_PER_CAPITA).
- **result.source:** `api` (live), `cache` (SQLite, TTL 24h), or `partial` (cache fallback).
- **Cache TTL:** Default 24h. Change with `BcchClient(ttl_seconds=3600)` for 1 hour.

In [ ]:
if os.environ.get('BCCH_TOKEN'):
    result = client.get(Series.USD, '2020-01-01', '2020-01-31')
    none_obs = [o for o in result.observations if o.value is None]
    print(f'USD Jan 2020: {len(result.observations)} total, {len(none_obs)} with value=None')
    if none_obs:
        print('Missing dates (value=None):')
        for o in none_obs[:5]:
            print(f'  {o.date}')
    else:
        print('All dates have values (no gaps in this range).')
else:
    print('Set BCCH_TOKEN to see None-handling in live data.')

## What you learned

- Fetch series with `client.get(Series.X, 'YYYY-MM-DD', 'YYYY-MM-DD')`
- Aggregate daily → monthly (month-end) for cleaner charts
- Dual-axis plots for different units/frequencies
- Bar charts for quarterly data (PIB)
- Multi-series plots (IMACEC original vs SA)
- Offline fallback when no token or API down
- Handle `value=None` (missing data, not zero)

**Next steps:**
- [README](https://github.com/cristobal437/econchile#readme)
- [API walkthrough notebook](econchile_walkthrough.ipynb)
- [PyPI](https://pypi.org/project/econchile/)